In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.5.1+cu121
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# Number of past daily maps
SEQ_LEN = 4

# Example only: AOD + mask + six auxiliary channels
INPUT_CHANNELS = 8

# Spatial CNN
CNN_CHANNELS = 32

# ConvLSTM hidden channels
HIDDEN_CHANNELS = 64

# Reconstruction head
HEAD_CHANNELS = 32

# Convolution kernel
KERNEL_SIZE = 3

print("Past days:", SEQ_LEN)
print("Input channels per day:", INPUT_CHANNELS)
print("Spatial CNN channels:", CNN_CHANNELS)
print("ConvLSTM hidden channels:", HIDDEN_CHANNELS)

Past days: 4
Input channels per day: 8
Spatial CNN channels: 32
ConvLSTM hidden channels: 64


In [3]:
class ConvLSTMCell(nn.Module):

    def __init__(
        self,
        input_channels,
        hidden_channels,
        kernel_size=3
    ):
        super().__init__()

        self.input_channels = input_channels
        self.hidden_channels = hidden_channels

        padding = kernel_size // 2

        # One convolution computes all four gate tensors.
        self.gates_conv = nn.Conv2d(
            in_channels=input_channels + hidden_channels,
            out_channels=4 * hidden_channels,
            kernel_size=kernel_size,
            padding=padding,
            bias=True
        )

        # Initialize forget-gate bias to 1.
        # Gate order: input, forget, output, candidate.
        with torch.no_grad():
            self.gates_conv.bias[
                hidden_channels:2 * hidden_channels
            ].fill_(1.0)

    def forward(self, x, state):

        # Current input: [B, C_in, H, W]
        # Previous hidden state: [B, C_hidden, H, W]
        # Previous cell state: [B, C_hidden, H, W]

        h_prev, c_prev = state

        # Concatenate current input and previous hidden state.
        combined = torch.cat(
            [x, h_prev],
            dim=1
        )

        # Shape: [B, 4 * hidden_channels, H, W]
        gates = self.gates_conv(combined)

        # Split into the four gate computations.
        i, f, o, g = torch.chunk(
            gates,
            chunks=4,
            dim=1
        )

        # Gate activation functions.
        i = torch.sigmoid(i)  # Input gate
        f = torch.sigmoid(f)  # Forget gate
        o = torch.sigmoid(o)  # Output gate
        g = torch.tanh(g)     # Candidate state

        # Update the cell state.
        c = f * c_prev + i * g

        # Calculate the new hidden state.
        h = o * torch.tanh(c)

        return h, c

    def init_state(self, x):

        # x: [B, C_in, H, W]
        batch_size, _, height, width = x.shape

        state_shape = (
            batch_size,
            self.hidden_channels,
            height,
            width
        )

        h0 = x.new_zeros(state_shape)
        c0 = x.new_zeros(state_shape)

        return h0, c0

In [4]:
# Example input: one batch, 32 channels, 64 x 64 grid
test_x = torch.randn(2, 32, 64, 64)

cell = ConvLSTMCell(
    input_channels=32,
    hidden_channels=64,
    kernel_size=3
)

initial_state = cell.init_state(test_x)

h, c = cell(test_x, initial_state)

print("Input shape:", test_x.shape)
print("Hidden state shape:", h.shape)
print("Cell state shape:", c.shape)

Input shape: torch.Size([2, 32, 64, 64])
Hidden state shape: torch.Size([2, 64, 64, 64])
Cell state shape: torch.Size([2, 64, 64, 64])


In [5]:
class ConvLSTMSequence(nn.Module):

    def __init__(
        self,
        input_channels,
        hidden_channels,
        kernel_size=3,
        return_sequences=True
    ):
        super().__init__()

        self.return_sequences = return_sequences

        self.cell = ConvLSTMCell(
            input_channels=input_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size
        )

    def forward(self, x):

        # Input: [B, T, C, H, W]
        batch_size, seq_len, _, height, width = x.shape

        # Initialize states using the first input frame.
        h, c = self.cell.init_state(x[:, 0])

        outputs = []

        # Process from the oldest day to the newest day.
        for t in range(seq_len):

            h, c = self.cell(
                x[:, t],
                (h, c)
            )

            outputs.append(h)

        if self.return_sequences:
            # [B, T, hidden_channels, H, W]
            return torch.stack(outputs, dim=1)

        # Only the final hidden state.
        # [B, hidden_channels, H, W]
        return h

In [6]:
# Four time steps, 32 input channels
test_sequence = torch.randn(
    2, 4, 32, 64, 64
)

sequence_layer = ConvLSTMSequence(
    input_channels=32,
    hidden_channels=64,
    return_sequences=True
)

sequence_output = sequence_layer(test_sequence)

print("Input sequence:", test_sequence.shape)
print("Output sequence:", sequence_output.shape)

Input sequence: torch.Size([2, 4, 32, 64, 64])
Output sequence: torch.Size([2, 4, 64, 64, 64])


In [7]:
class StackedConvLSTM(nn.Module):

    def __init__(
        self,
        input_channels,
        cnn_channels=32,
        hidden_channels=64,
        head_channels=32,
        kernel_size=3
    ):
        super().__init__()

        # ------------------------------------------------
        # Layer 1: Spatial feature extraction
        # ------------------------------------------------
        self.spatial_cnn = nn.Sequential(
            nn.Conv2d(
                in_channels=input_channels,
                out_channels=cnn_channels,
                kernel_size=kernel_size,
                padding=kernel_size // 2
            ),
            nn.GELU()
        )

        # ------------------------------------------------
        # Layer 2: First ConvLSTM
        # ------------------------------------------------
        self.convlstm1 = ConvLSTMSequence(
            input_channels=cnn_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size,
            return_sequences=True
        )

        # ------------------------------------------------
        # Layer 3: Second ConvLSTM
        # ------------------------------------------------
        self.convlstm2 = ConvLSTMSequence(
            input_channels=hidden_channels,
            hidden_channels=hidden_channels,
            kernel_size=kernel_size,
            return_sequences=False
        )

        # ------------------------------------------------
        # Layer 4: Spatial reconstruction head
        # ------------------------------------------------
        self.reconstruction_head = nn.Sequential(
            nn.Conv2d(
                hidden_channels,
                head_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GELU(),

            nn.Conv2d(
                head_channels,
                1,
                kernel_size=1
            )
        )

    def forward(self, x):

        # x: [B, T, C, H, W]
        batch_size, seq_len, channels, height, width = x.shape

        # Apply the same CNN to every input day.
        x = x.reshape(
            batch_size * seq_len,
            channels,
            height,
            width
        )

        x = self.spatial_cnn(x)

        # Restore the time dimension.
        x = x.reshape(
            batch_size,
            seq_len,
            -1,
            height,
            width
        )

        # First ConvLSTM returns every hidden state.
        x = self.convlstm1(x)

        # Second ConvLSTM returns the final hidden state.
        x = self.convlstm2(x)

        # Predict the current day's AOD map.
        prediction = self.reconstruction_head(x)

        return prediction

In [13]:
model = StackedConvLSTM(
    input_channels=INPUT_CHANNELS,
    cnn_channels=CNN_CHANNELS,
    hidden_channels=HIDDEN_CHANNELS,
    head_channels=HEAD_CHANNELS,
    kernel_size=KERNEL_SIZE
).to(device)

In [9]:
print(model)

total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\nTotal trainable parameters:", f"{total_params:,}")

print("\nParameter count by component:")

for name, module in model.named_children():
    params = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    print(f"{name:25s}: {params:,}")

StackedConvLSTM(
  (spatial_cnn): Sequential(
    (0): Conv2d(8, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GELU(approximate='none')
  )
  (convlstm1): ConvLSTMSequence(
    (cell): ConvLSTMCell(
      (gates_conv): Conv2d(96, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
  )
  (convlstm2): ConvLSTMSequence(
    (cell): ConvLSTMCell(
      (gates_conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
  )
  (reconstruction_head): Sequential(
    (0): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(32, 1, kernel_size=(1, 1), stride=(1, 1))
  )
)

Total trainable parameters: 537,441

Parameter count by component:
spatial_cnn              : 2,336
convlstm1                : 221,440
convlstm2                : 295,168
reconstruction_head      : 18,497


In [14]:
def masked_mse(prediction, target, holdout_mask):

    # Ensure mask and predictions use compatible types.
    weights = holdout_mask.to(
        dtype=prediction.dtype,
        device=prediction.device
    )

    # Calculate squared error.
    squared_error = (prediction - target).square()

    # Only artificially hidden, originally valid pixels
    # contribute to the loss.
    numerator = (squared_error * weights).sum()

    denominator = weights.sum().clamp_min(1.0)

    return numerator / denominator